# 네이버 검색 OpenAPI 기반 쇼핑 데이터 수집 노트북

이 노트북은 globalgates 카테고리 분류 모델 학습용 외부 데이터를 **네이버 검색 OpenAPI(쇼핑)**로 수집한다.

## 왜 직접 크롤링이 아니라 OpenAPI인가
- 네이버 쇼핑 페이지는 봇 차단이 매우 강해서 stealth/headed 모드도 캡차로 막힘
- 쿠팡은 Akamai로 모든 자동 접근이 Access Denied
- 네이버 검색 OpenAPI는 합법, 무료, 일일 25,000건 호출 가능
- 응답에 `category1~4`가 포함돼서 카테고리 라벨이 깨끗함

## 이 노트북의 범위
- API 키 로드 검증
- 카테고리별 검색어 사전 정의
- API 응답 구조 확인 (단건 호출)
- 페이징 수집 함수
- 카테고리별 수집 + CSV 저장

## 컨벤션
- `machine-learning` 커널
- `.env`에서 키 로드 (하드코딩 금지)
- 주석은 "왜 이 셀을 실행하는지" 중심


## 1. 라이브러리 + 키 로드

키가 없으면 뒤 셀이 다 의미 없으니 가장 먼저 점검한다.

In [ ]:
from pathlib import Path
import os
import time
import json

import pandas as pd
import requests
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
ENV_PATH = PROJECT_ROOT / ".env"

load_dotenv(ENV_PATH)

CLIENT_ID = os.getenv("NAVER_OPENAPI_CLIENT_ID")
CLIENT_SECRET = os.getenv("NAVER_OPENAPI_CLIENT_SECRET")

if not CLIENT_ID or not CLIENT_SECRET:
    raise ValueError(
        ".env 파일에 NAVER_OPENAPI_CLIENT_ID / NAVER_OPENAPI_CLIENT_SECRET 가 비어있음. "
        "https://developers.naver.com/apps/#/list 에서 애플리케이션 등록 후 키를 .env에 추가하기."
    )

print("키 로드 OK (값은 출력하지 않음)")

## 2. globalgates 카테고리 → 검색어 매핑

네이버 쇼핑 검색 API는 검색어 1개당 최대 1,000건(100건 × 10페이지)까지 페이징 가능하다.
따라서 카테고리당 여러 검색어를 두고 수집해야 충분한 양과 다양성을 확보할 수 있다.

매핑 가능한 7개 globalgates 카테고리만 다룬다.
수출/수입/물류/관세/금융 5개는 일반 쇼핑몰에 매칭되는 상품이 거의 없어서 별도(합성 데이터)로 처리할 예정.

In [ ]:
CATEGORY_QUERIES = {
    "식품": ["라면", "과자", "음료", "즉석식품", "냉동식품", "김치", "반찬", "차", "커피원두", "양념"],
    "화장품": ["스킨", "로션", "선크림", "마스카라", "립스틱", "파운데이션", "클렌징폼", "향수", "마스크팩", "샴푸"],
    "자동차": ["자동차커버", "와이퍼", "카매트", "차량용청소기", "블랙박스", "차량용방향제", "차량용공기청정기", "타이어", "자동차왁스", "엔진오일"],
    "섬유/의류": ["티셔츠", "청바지", "원피스", "자켓", "코트", "운동복", "양말", "속옷", "잠옷", "정장"],
    "IT": ["노트북", "모니터", "키보드", "마우스", "무선이어폰", "스피커", "USB메모리", "외장하드", "공유기", "스마트폰케이스"],
    "기계/장비": ["전동드릴", "그라인더", "용접기", "드릴비트", "절단기", "에어컴프레서", "발전기", "사다리", "작업등", "측정공구"],
    "에너지": ["보조배터리", "리튬배터리", "태양광패널", "고속충전기", "전기히터", "LED전구", "무선충전기", "차량용충전기", "USB충전기", "충전케이블"],
}

total_queries = sum(len(v) for v in CATEGORY_QUERIES.values())
print(f"카테고리 수: {len(CATEGORY_QUERIES)}")
print(f"총 검색어 수: {total_queries}")
print(f"이론적 최대 수집량(검색어당 1000건 가정): {total_queries * 1000:,}")

In [ ]:
expected_id = "0oB7BsgKHMNSQzsqDtUZ"
print("일치:", CLIENT_ID == expected_id)
for i, (a, b) in enumerate(zip(CLIENT_ID, expected_id)):
  if a != b:
      print(f"  pos {i}: env={a!r} ({hex(ord(a))}) vs site={b!r} ({hex(ord(b))})")

## 3. 단건 호출로 응답 구조 확인

본 수집 전에 한 번만 호출해서 응답이 정상이고 어떤 필드가 들어오는지 본다.
여기서 키 권한 문제나 헤더 오타를 미리 잡는다.

In [ ]:
def fetch_naver_shop(query: str, display: int = 10, start: int = 1) -> dict:
    url = "https://openapi.naver.com/v1/search/shop.json"
    headers = {
        "X-Naver-Client-Id": CLIENT_ID,
        "X-Naver-Client-Secret": CLIENT_SECRET,
    }
    params = {"query": query, "display": display, "start": start, "sort": "sim"}
    res = requests.get(url, headers=headers, params=params, timeout=15)
    res.raise_for_status()
    return res.json()

probe = fetch_naver_shop("라면", display=3)
print("총 매칭:", probe.get("total"))
print("받은 건수:", len(probe.get("items", [])))
print("첫 건 키:", list(probe["items"][0].keys()) if probe.get("items") else "없음")
print()
print(json.dumps(probe["items"][0] if probe.get("items") else {}, ensure_ascii=False, indent=2))

## 4. 페이징 수집 함수

검색어 하나에 대해 100건 × 10페이지 = 최대 1,000건을 모은다.

- API는 `start` 1부터 시작, 한 번에 `display` 최대 100건
- 호출 사이 sleep을 둬서 API 정책 안전선 유지(공식 한도는 일일 25,000건이지만 burst 방지)
- 실패하면 해당 검색어만 건너뛰고 계속 진행

In [ ]:
def collect_query(query: str, max_items: int = 1000, sleep_sec: float = 0.3) -> list[dict]:
    collected = []
    per_page = 100
    for page in range(10):
        start = page * per_page + 1
        if start > 1000:
            break
        try:
            data = fetch_naver_shop(query, display=per_page, start=start)
        except requests.HTTPError as e:
            print(f"  [{query}] start={start} HTTP 에러: {e}, 이 검색어 중단")
            break
        items = data.get("items", [])
        if not items:
            break
        collected.extend(items)
        if len(collected) >= max_items:
            collected = collected[:max_items]
            break
        time.sleep(sleep_sec)
    return collected

sample = collect_query("라면", max_items=200)
print(f"라면 검색 수집 건수: {len(sample)}")
print(f"category1 분포 상위 5개:")
print(pd.Series([it["category1"] for it in sample]).value_counts().head())

## 5. 카테고리별 본 수집

전체 7카테고리 × 10 검색어를 순회한다.
각 검색어당 1,000건씩 시도해서 globalgates 카테고리 라벨과 함께 저장한다.

수집은 시간이 걸리므로 진행 상황을 카테고리/검색어 단위로 출력한다.
중간에 실패한 검색어는 스킵하고 끝까지 돈다.

In [ ]:
def parse_item(item: dict, query: str, gg_category: str) -> dict:
    return {
        "source": "naver_openapi",
        "query_keyword": query,
        "category_globalgates": gg_category,
        "category_site_l1": item.get("category1", ""),
        "category_site_l2": item.get("category2", ""),
        "category_site_l3": item.get("category3", ""),
        "category_site_l4": item.get("category4", ""),
        "title": item.get("title", "").replace("<b>", "").replace("</b>", ""),
        "brand": item.get("brand", ""),
        "maker": item.get("maker", ""),
        "mall_name": item.get("mallName", ""),
        "product_id": item.get("productId", ""),
        "lprice": item.get("lprice", ""),
        "hprice": item.get("hprice", ""),
        "link": item.get("link", ""),
    }

records = []
for gg_category, queries in CATEGORY_QUERIES.items():
    print(f"=== {gg_category} ===")
    for q in queries:
        items = collect_query(q, max_items=1000)
        records.extend(parse_item(it, q, gg_category) for it in items)
        print(f"  {q}: {len(items)}건 누적 {len(records):,}")

df = pd.DataFrame(records)
print()
print(f"총 수집: {len(df):,}건")

## 6. 빠른 품질 점검

- 카테고리별 분포가 한쪽으로 쏠렸는지
- API의 category1과 globalgates 카테고리가 잘 매칭되는지(노이즈 비율 감 잡기)
- 중복 productId 비율

In [ ]:
print("globalgates 카테고리별 건수:")
print(df["category_globalgates"].value_counts())
print()
print("중복 productId 비율:", round((df["product_id"].duplicated().sum() / len(df)) * 100, 2), "%")
print()
print("globalgates 카테고리별 네이버 category1 매칭 상위:")
for gg in df["category_globalgates"].unique():
    sub = df[df["category_globalgates"] == gg]
    print(f"\n--- {gg} ---")
    print(sub["category_site_l1"].value_counts().head(5))

## 7. CSV로 저장

`datasets/naver_openapi_raw.csv`로 저장. 다음 노트북에서는 이 CSV를 출발점으로 라벨 정제 + 학습 데이터 구성을 진행한다.

In [ ]:
OUT_PATH = PROJECT_ROOT / "datasets" / "naver_openapi_raw.csv"
df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print(f"저장 완료: {OUT_PATH}")
print(f"파일 크기: {OUT_PATH.stat().st_size / 1024:.1f} KB")

## 다음 단계

1. `03_db_dataset_eda.ipynb` — 실서비스 DB의 카테고리 라벨 데이터 양/분포 확인
2. `04_label_cleanup.ipynb` — 네이버 category1과 globalgates 카테고리 매핑 룰 정제, 노이즈 제거
3. `05_baseline_classification.ipynb` — TF-IDF + LogReg/LinearSVC 베이스라인
4. 부족한 카테고리(수출/수입/물류/관세/금융)는 합성 데이터로 별도 보강